In [ ]:
import mpl_axes_aligner
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pandas as pd
import seaborn as sns
import os
os.makedirs('Figures', exist_ok=True)

In [ ]:
X = pd.read_csv('X_Clean.csv' , index_col = 0)
y = pd.read_csv('Y_Clean.csv')['Bankrupt?']
print(y.shape)
print(X.columns)

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=20, random_state=45735476)
components = pca.fit_transform(X)

#Number of components to be retained using Kaiser's Criterion
#Use the mean eigenvalue, since quantile transform with uniform distribution was used.
thres = sum(pca.explained_variance_)/64

n = 0
for i in range(pca.explained_variance_.shape[0]):
    if pca.explained_variance_[i] > thres:
        print (pca.explained_variance_[i])
        n = n + 1

print(n, 'components are to be retained.', 'with total variance explained of: ', sum(pca.explained_variance_ratio_[:n])*100, '%')

plt.figure(figsize=(10, 8))
plt.bar(range(1,len(pca.explained_variance_ratio_)+1), pca.explained_variance_ratio_)
plt.ylabel('Explained Variance')
plt.xlabel('Principal Components')
plt.plot(range(1,len(pca.explained_variance_ratio_)+1),
         np.cumsum(pca.explained_variance_ratio_),
         color='blue',
         label='Cumulative Explained Variance')
plt.axvline(x=15, color='r', label = 'Cutoff for variables retained')
plt.legend(loc='best')
plt.savefig('Figures/Cumulative Explained Variance.png', dpi=300)
plt.show()


In [ ]:
#Here we create the full on biplot.
idx = np.argsort(y)
def biplot(score, coeff, labels=None):
    xs = score[:, 0]
    ys = score[:, 1]
    plt.figure(figsize=(10, 8))
    plt.scatter(xs, ys, c=y, cmap='coolwarm')
    plt.colorbar(label='Bankruptcies')

    lengths = np.sqrt(coeff[:, 0]**2 + coeff[:, 1]**2)
    top = np.argsort(lengths)[-3:]

    factor = max(np.abs(xs).max(), np.abs(ys).max()) / lengths.max()

    for j in top:

        plt.arrow(0, 0, coeff[j, 0]*factor, coeff[j, 1]*factor, color='w', alpha=0.5, head_width=0.1, head_length=0.1)
        plt.text(coeff[j, 0] * factor, coeff[j, 1] * factor, labels[j], color='w', horizontalalignment='left', verticalalignment='center')


    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title("Biplot")
    plt.savefig('Figures/PCA Biplot.png', dpi=300, bbox_inches='tight')
    plt.show()

biplot(components, pca.components_.T, X.columns)

In [ ]:
#3D plot to see if that helps to better visualize the data
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(10, 12))
ax = fig.add_subplot(111, projection='3d')

scatter_3d =ax.scatter(components[idx, 0], components[idx, 1], components[idx, 2], c=y.iloc[idx], cmap='coolwarm')

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')

plt.colorbar(scatter_3d, label='Bankruptcies', shrink=0.7)
plt.savefig('Figures/3D Exploratory Plot.png', dpi=300)
plt.show()

In [ ]:
#Loadings to see which factors influence the most here.
for x in range(2):
    loading_scores = pd.Series(pca.components_[x], index = X.columns)
    sorted_loading_scores = loading_scores.abs().sort_values(ascending = False)
    top_10_factors = sorted_loading_scores.index[:10].values
    print( 'Most influential factors for PC', x+1,'\n', loading_scores[top_10_factors] , '\n\n')


